<a href="https://colab.research.google.com/github/MalavMDesai/GenAIAssignment/blob/main/Assignment5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:

!pip install -q langchain-community
!pip install -q pymupdf
!pip install -q langchain-google-genai
!pip install -q chromadb
!pip install -q langchain-openai

In [15]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [16]:
os.environ["GOOGLE_API_KEY"] = userdata.get('Gemini_key')

# policy_file_url = "https://github.com/MalavMDesai/GenAIAssignment/raw/refs/heads/main/Assignment5_Policy.pdf"
policy_file_url="https://customer-portal-assets.hdfcergo.com/documents/OptimaPlus-192946032259.pdf"
# policy_file_url = "https://github.com/MalavMDesai/GenAIAssignment/raw/refs/heads/main/Assignment5_Policy-1-5.pdf"

loader = PyMuPDFLoader(policy_file_url)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)

In [17]:
print(len(chunks))
print(chunks[0])

84
page_content='OPTIMA PLUS - Prospectus 
HDFC ERGO General Insurance Limited 
 
HDFC ERGO General Insurance Company Limited. IRDAI Reg. No.146. CIN: U66030MH2007PLC177117. Registered & 
Corporate Office: 6th Floor, Leela Business Park, Andheri-Kurla Road, Andheri (East), Mumbai – 400 059. UIN: 
Optima Plus - HDHHLIP21336V022021  
 
1 
Optima Plus - Prospectus 
Eligibility 
▪ 
This policy covers persons in the age group 91 days to 65 years.  
▪ 
The maximum entry age is restricted upto 65 years. 
▪ 
Child between 91 days to 5 years can be insured only when either parent is getting 
insured under this policy.  
▪ 
The policy offers coverage on individual sum insured basis.  
▪ 
This policy can be issued to an individual and/or family. 
▪' metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-05-06T10:28:39+05:30', 'source': 'https://customer-portal-assets.hdfcergo.com/documents/OptimaPlus-192946032259.pdf', 'fil

In [18]:
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", task_type="retrieval_document")
vector_store = Chroma.from_documents(chunks, embedding)
retriever = vector_store.as_retriever()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.0)#, override_model_name=True)

system_prompt = (
    "You work as a expert Insurance Claims agent in HDFC ERGO General Insurance Company Limited \n"
    "Answer the questions using only provided policy context. \n"
    "If you do not know the answer or if it is not is the provided document than exactly say:\n"
    "'Unable to find the information in policy document, connect to customer care'"
    "CONTEXT:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', "{input}")
])

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# query1 = "What is the waiting period for pre-existing diseases?"
# res = rag_chain.invoke(query1)

# print(f"User query: {query1}")
# print(f"ans: \n {res}")


In [19]:
policy_queries = [
    "What is the entry age limit?",
    "Is maternity covered?",
    "What is the waiting period for pre-existing diseases?",
    "What documents are required for filing a claim?"
]
for index, query in enumerate(policy_queries, start=1):
  res = rag_chain.invoke(query)
  print(f"Processing Query #{index}: {query}")
  print(f"Rag Answers #{index}: \n {res} \n")

Processing Query #1: What is the entry age limit?
Rag Answers #1: 
 Based on the provided policy document, the entry age limit is from 91 days to 65 years, with the maximum entry age restricted up to 65 years. Additionally, a child between 91 days to 5 years can be insured only when either parent is getting insured under the policy. 

Processing Query #2: Is maternity covered?
Rag Answers #2: 
 Based on the provided policy document, maternity is generally excluded under **Code – Excl18**. 

Specifically, the following are excluded:
* **Medical treatment expenses traceable to childbirth** (including complicated deliveries and caesarean sections incurred during hospitalization), **except ectopic pregnancy**.
* **Expenses towards miscarriage** (**unless due to an accident**) and lawful medical termination of pregnancy during the Policy period. 

Processing Query #3: What is the waiting period for pre-existing diseases?
Rag Answers #3: 
 Based on the provided policy document, the waiting p